In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("andradaolteanu/gtzan-dataset-music-genre-classification")

print("Path to dataset files:", path)

Path to dataset files: /home/nikita/.cache/kagglehub/datasets/andradaolteanu/gtzan-dataset-music-genre-classification/versions/1


In [2]:
import os

file = os.listdir(path)
files_in_dir = os.listdir(path + '/' + file[0])

print(files_in_dir)

path_to_files = path + '/' + file[0]

base_path = path_to_files + '/' + files_in_dir[3]

files_list = os.listdir(base_path)

data_files = []

for i in range(len(files_list)):
    class_path = os.path.join(base_path, files_list[i])
    path_obj = os.listdir(class_path)
    path_obj_full = [os.path.join(class_path, f) for f in path_obj]
    data_files.append(path_obj_full)

['genres_original', 'features_3_sec.csv', 'features_30_sec.csv', 'images_original']


In [3]:
import pandas as pd
import numpy as np
from PIL import Image
import random

data_x = []
data_y = []
size = (260, 260)

for class_idx, file_list in enumerate(data_files):
    num_samples = len(file_list)
    
    for j in range(num_samples):
        img = Image.open(file_list[j])
        res = img.convert('RGB').resize(size)
        res = np.array(res) / 255.0
        v_stripes = np.hsplit(res, 10)
        data_x.append(v_stripes)
        data_y.append(class_idx)

In [4]:
from sklearn.model_selection import train_test_split

data_x = np.array(data_x)
data_y = np.array(data_y)

x_train, x_test, y_train, y_test = train_test_split(
    data_x, 
    data_y, 
    test_size=0.10, 
    random_state=42,
    stratify=data_y
)

In [5]:
del data_x
del data_y

In [7]:
import tensorflow as tf
import keras

print("Версия TensorFlow:", tf.__version__)
print("Доступные GPU:", tf.config.list_physical_devices('GPU'))
print("Устройство для вычислений:", tf.test.is_gpu_available())

Версия TensorFlow: 2.14.0
Доступные GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Устройство для вычислений: True


2026-05-06 20:57:44.537745: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-05-06 20:57:44.539167: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-05-06 20:57:44.539982: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysf

In [9]:
pip install tensorflow_model_optimization

/bin/bash: /home/nikita/anaconda3/envs/tf-gpu/lib/libtinfo.so.6: no version information available (required by /bin/bash)
Note: you may need to restart the kernel to use updated packages.


In [25]:
from tensorflow.keras.callbacks import EarlyStopping
from keras.layers import Conv2D, MaxPooling2D, Flatten
from tensorflow.keras import layers

class SelfAttention(tf.keras.layers.Layer):
    def __init__(self, use_scale=False, **kwargs):
        super().__init__(**kwargs)
        self.use_scale = use_scale
        self.attention = Attention(use_scale=use_scale)
    
    def build(self, input_shape):
        self.W = self.add_weight(
            name='att_weight',
            shape=(input_shape[-1], 1),
            initializer='glorot_uniform',
            trainable=True
        )
        super().build(input_shape)

    def call(self, x):
        # x: (batch, timesteps, features)
        e = tf.matmul(x, self.W)                # (batch, timesteps, 1)
        e = tf.squeeze(e, axis=-1)              # (batch, timesteps)
        alpha = tf.nn.softmax(e, axis=-1)       # веса внимания (batch, timesteps)
        context = tf.reduce_sum(x * tf.expand_dims(alpha, -1), axis=1)  # (batch, features)
        return context

model = keras.Sequential([
    keras.Input(shape = (10, 260, 26, 3)),
    keras.layers.TimeDistributed(Conv2D(5, (3, 3), padding = 'same', strides = 2)),
    keras.layers.Dropout(0.25),
    keras.layers.TimeDistributed(MaxPooling2D((2, 4))),
    keras.layers.TimeDistributed(Conv2D(5, (3, 3), padding = 'same', activation ='relu', strides = 2)),
    keras.layers.Dropout(0.25),
    keras.layers.TimeDistributed(Flatten()),
    keras.layers.GRU(32, return_sequences=True, activation='tanh'),
    SelfAttention(use_scale=True),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.3),
    Dense(256, activation='relu'),
    keras.layers.Dropout(0.3),
    Dense(128),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(10, activation ='softmax')
])

model.compile(  optimizer = keras.optimizers.Adam(learning_rate = 1e-4),
                loss = 'sparse_categorical_crossentropy',
                metrics = ['accuracy']
                )

early_stopping = EarlyStopping(
                    monitor='val_loss',
                    mode='min',
                    patience = 50,
                    min_delta = 0.01,
                    verbose = 0,
                    restore_best_weights = True
                    )


In [26]:
model.fit(
        x = x_train,
        y = y_train,
        batch_size = 16,
        epochs = 2000,
        verbose = 0,
        callbacks = [early_stopping], 
        shuffle = True,
        validation_split = 0.1,
        validation_data = None,
        validation_batch_size = 16
        )

2026-05-06 21:26:59.495706: W tensorflow/core/grappler/costs/op_level_cost_estimator.cc:693] Error in PredictCost() for the op: op: "Softmax" attr { key: "T" value { type: DT_FLOAT } } inputs { dtype: DT_FLOAT shape { unknown_rank: true } } device { type: "GPU" vendor: "NVIDIA" model: "NVIDIA GeForce RTX 4070 SUPER" frequency: 2475 num_cores: 56 environment { key: "architecture" value: "8.9" } environment { key: "cuda" value: "11080" } environment { key: "cudnn" value: "8600" } num_registers: 65536 l1_cache_size: 24576 l2_cache_size: 50331648 shared_memory_size_per_multiprocessor: 102400 memory_size: 10225975296 bandwidth: 504048000 } outputs { dtype: DT_FLOAT shape { unknown_rank: true } }
2026-05-06 21:27:00.837541: W tensorflow/core/grappler/costs/op_level_cost_estimator.cc:693] Error in PredictCost() for the op: op: "Softmax" attr { key: "T" value { type: DT_FLOAT } } inputs { dtype: DT_FLOAT shape { unknown_rank: true } } device { type: "GPU" vendor: "NVIDIA" model: "NVIDIA GeForc

In [27]:
train_loss, keras_train_acc = model.evaluate(x_train, y_train)
test_loss, keras_test_acc = model.evaluate(x_test, y_test)
print('\nдля CNN+RNN точность для обучающей \ тестовой выборки:', round(keras_train_acc, 2), ' \ ', round(keras_test_acc, 2))

4/4 [==============================] - 0s 4ms/step - loss: 1.2643 - accuracy: 0.6300

для CNN+RNN точность для обучающей \ тестовой выборки: 0.63  \  0.63


In [28]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)

converter.optimizations = [tf.lite.Optimize.DEFAULT]

def representative_dataset_gen():
    for i in range(100):
        yield [x_train[i:i+1].astype(np.float32)]

converter.representative_dataset = representative_dataset_gen

converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,       
    tf.lite.OpsSet.SELECT_TF_OPS    
]
converter._experimental_lower_tensor_list_ops = False  

converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8

tflite_model_int8 = converter.convert()

with open('model_quantized_int8.tflite', 'wb') as f:
    f.write(tflite_model_int8)

print("✅ Модель успешно сконвертирована с SELECT_TF_OPS и lower_tensor_list_ops = False")

INFO:tensorflow:Assets written to: /tmp/tmpygmhwvq0/assets


INFO:tensorflow:Assets written to: /tmp/tmpygmhwvq0/assets
/home/nikita/anaconda3/envs/tf-gpu/lib/python3.10/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-05-06 21:30:40.209406: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-06 21:30:40.209421: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-06 21:30:40.209524: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpygmhwvq0
2026-05-06 21:30:40.215587: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-06 21:30:40.215601: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpygmhwvq0
2026-05-06 21:30:40.231044: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05

✅ Модель успешно сконвертирована с SELECT_TF_OPS и lower_tensor_list_ops = False


fully_quantize: 0, inference_type: 6, input_inference_type: UINT8, output_inference_type: UINT8


In [29]:
interpreter = tf.lite.Interpreter(model_path='model_quantized_int8.tflite')
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]

def predict_tflite(x):
    if input_details['dtype'] == np.uint8:
        input_scale, input_zero_point = input_details['quantization']
        x_quant = (x / input_scale + input_zero_point).astype(np.uint8)
    else:
        x_quant = x.astype(np.float32)
    interpreter.set_tensor(input_details['index'], x_quant)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details['index'])
    if output_details['dtype'] == np.uint8:
        output_scale, output_zero_point = output_details['quantization']
        output = (output.astype(np.float32) - output_zero_point) * output_scale
    return output

correct = 0
for i in range(len(x_test)):
    pred = predict_tflite(x_test[i:i+1])
    if np.argmax(pred) == y_test[i]:
        correct += 1
print(f"Quantized accuracy: {correct/len(x_test):.4f}")

Quantized accuracy: 0.6200


2026-05-06 21:30:40.808800: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-05-06 21:30:40.809762: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-05-06 21:30:40.810538: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysf